# Flow Map Matching & Lagrangian Self-Distillation (1-D CPU Demo)

This self-contained notebook demonstrates both two-time flow-map workflows in
`swirl_dynamics.lib.flow_map`:

1. **Teacher Velocity Field:** Fits a lightweight `flow_matching.RescaledUnet`
   velocity field $v_\phi(x_t, t)$ along `RectifiedFlow()` on a 1-D periodic
   waveform dataset and generates a 32-step Euler ODE reference.
2. **Teacher-Based Distillation (`LagrangianFlowMapModel`):** Distills $v_\phi$
   into a two-time flow map $X_{s, t}^\theta$ (`RescaledFlowMapUNet`) with exact
   boundary condition $X_{s, s}^\theta(x) = x$.
3. **Teacher-Free Self-Distillation (`LagrangianSelfDistilledFlowMapModel`):**
   Trains $X_{s, t}^\theta(x) = x + (t - s) u_\theta(x, t, s)$ directly from
   samples without a teacher and compares 1-, 2-, and 4-NFE forward maps via
   `flow_map.sample_forward_flow_map`.


## 1. Installation & Imports


In [ ]:
!pip install -q "flax>=0.12.8" git+https://github.com/google-research/swirl-dynamics.git

In [ ]:
import os
import tempfile

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
from swirl_dynamics.lib import flow_map
from swirl_dynamics.lib import flow_matching
from swirl_dynamics.projects.distillation.flow_map_matching.configs import config_1d

## 2. Configuration


In [ ]:
config = config_1d.get_config()
workdir = tempfile.mkdtemp(prefix="flow_map_demo_")

## 3. Synthetic 1-D Periodic Waveform Dataset


In [ ]:
def make_waveforms(num_samples, grid_size, seed):
  rng = np.random.default_rng(seed)
  grid = np.linspace(0.0, 2.0 * np.pi, grid_size, endpoint=False)
  phase = rng.uniform(0.0, 2.0 * np.pi, size=(num_samples, 1))
  amp = rng.uniform(0.8, 1.2, size=(num_samples, 1))
  x_1 = amp * np.sin(grid[None, :] + phase) + 0.3 * np.cos(
      2.0 * grid[None, :] - phase
  )
  x_1 = x_1[..., None].astype(np.float32)
  x_0 = rng.normal(scale=0.8, size=x_1.shape).astype(np.float32)
  return {"x_0": x_0, "x_1": x_1}


train_data = make_waveforms(
    config.num_train_samples, config.grid_size, seed=config.seed
)
eval_data = make_waveforms(
    config.num_eval_samples, config.grid_size, seed=config.seed + 1
)


def batch_iterator(dataset, batch_size, seed):
  rng = np.random.default_rng(seed)
  n = dataset["x_0"].shape[0]
  while True:
    idx = rng.integers(0, n, size=(batch_size,))
    yield {k: v[idx] for k, v in dataset.items()}

## 4. Train Lightweight `lib.flow_matching` Teacher Velocity


In [ ]:
input_shape = (config.grid_size, config.num_channels)
interp = flow_matching.RectifiedFlow()

teacher_net = flow_matching.RescaledUnet(
    out_channels=config.num_channels,
    num_channels=config.num_channels_unet,
    downsample_ratio=config.downsample_ratio,
    num_blocks=config.num_blocks,
    noise_embed_dim=config.noise_embed_dim,
    use_attention=False,
    use_position_encoding=False,
)
teacher_model = flow_matching.StochasticInterpolantModel(
    input_shape=input_shape,
    flow_model=teacher_net,
    interpolant=interp,
    loss_stochastic_interpolant=flow_matching.velocity_loss,
)

teacher_vars = teacher_model.initialize(jax.random.PRNGKey(config.seed))
teacher_params = teacher_vars["params"]
teacher_opt = optax.adam(config.learning_rate)
teacher_opt_state = teacher_opt.init(teacher_params)


@jax.jit
def teacher_step(params, opt_state, batch, rng):
  (loss, _), grads = jax.value_and_grad(teacher_model.loss_fn, has_aux=True)(
      params, batch, rng, {}
  )
  updates, next_opt = teacher_opt.update(grads, opt_state, params)
  return optax.apply_updates(params, updates), next_opt, loss


t_iter = batch_iterator(train_data, config.batch_size, seed=config.seed + 2)
teacher_rng = jax.random.PRNGKey(config.seed + 3)
teacher_loss = 0.0
for _ in range(config.teacher_steps):
  teacher_rng, step_rng = jax.random.split(teacher_rng)
  teacher_params, teacher_opt_state, loss_val = teacher_step(
      teacher_params, teacher_opt_state, next(t_iter), step_rng
  )
  teacher_loss = float(loss_val)

teacher_fn = flow_matching.StochasticInterpolantModel.inference_fn(
    {"params": teacher_params}, teacher_net
)
print(f"Final teacher velocity loss: {teacher_loss:.4f}")

## 5. Train Teacher-Distilled Flow Map & Teacher-Free LSD Flow Map


In [ ]:
fm_net = flow_map.RescaledFlowMapUNet(
    out_channels=config.num_channels,
    num_channels=config.num_channels_unet,
    downsample_ratio=config.downsample_ratio,
    num_blocks=config.num_blocks,
    noise_embed_dim=config.noise_embed_dim,
    use_attention=False,
    use_position_encoding=False,
)
distill_model = flow_map.LagrangianFlowMapModel(
    input_shape=input_shape,
    flow_map_model=fm_net,
    flow_model=teacher_net,
    params_flow=teacher_params,
    interpolant=interp,
    number_of_eval_steps=config.eval_num_steps,
)
distill_trainer = flow_map.LagrangianFlowMapTrainer(
    ema_decay=config.ema_decay,
    model=distill_model,
    rng=jax.random.PRNGKey(config.seed + 10),
    optimizer=optax.adam(config.learning_rate),
)
distill_metrics = distill_trainer.train(
    batch_iterator(train_data, config.batch_size, seed=config.seed + 11),
    num_steps=config.distill_steps,
).compute()

lsd_net = flow_map.RescaledMeanFlowUNet(
    out_channels=config.num_channels,
    num_channels=config.num_channels_unet,
    downsample_ratio=config.downsample_ratio,
    num_blocks=config.num_blocks,
    noise_embed_dim=config.noise_embed_dim,
    use_attention=False,
    use_position_encoding=False,
)
lsd_model = flow_map.LagrangianSelfDistilledFlowMapModel(
    input_shape=input_shape,
    mean_flow_model=lsd_net,
    interpolant=interp,
    weight_v=config.lsd_weight_v,
    number_of_eval_steps=config.eval_num_steps,
)
lsd_trainer = flow_map.LagrangianSelfDistilledFlowMapTrainer(
    ema_decay=config.ema_decay,
    model=lsd_model,
    rng=jax.random.PRNGKey(config.seed + 20),
    optimizer=optax.adam(config.learning_rate),
)
lsd_metrics = lsd_trainer.train(
    batch_iterator(train_data, config.batch_size, seed=config.seed + 21),
    num_steps=config.lsd_steps,
).compute()
print(
    f"Distilled loss: {float(distill_metrics['train_loss']):.4f} | "
    f"LSD loss: {float(lsd_metrics['train_loss']):.4f}"
)

## 6. Compare 1/2/4-NFE Flow Maps vs. 32-Step Euler Teacher ODE


In [ ]:
x_eval_0 = jnp.asarray(eval_data["x_0"])


def teacher_euler_ode(x_init, num_steps):
  dt = 1.0 / num_steps
  x = x_init
  for k in range(num_steps):
    t = jnp.full((x.shape[0],), k * dt, dtype=jnp.float32)
    x = x + dt * teacher_fn(x, t)
  return x


ode_ref = np.asarray(teacher_euler_ode(x_eval_0, config.teacher_ode_steps))
distill_fn = flow_map.LagrangianFlowMapTrainer.inference_fn_from_state_dict(
    distill_trainer.train_state, fm_net, use_ema=True
)
lsd_fn = (
    flow_map.LagrangianSelfDistilledFlowMapTrainer.inference_fn_from_state_dict(
        lsd_trainer.train_state, lsd_net, use_ema=True
    )
)

metrics = {
    "teacher_loss": teacher_loss,
    "distill_loss": float(distill_metrics["train_loss"]),
    "lsd_loss": float(lsd_metrics["train_loss"]),
}
distill_samples = {}
lsd_samples = {}
for nfe in config.eval_num_steps:
  distill_samples[nfe] = np.asarray(
      flow_map.sample_forward_flow_map(distill_fn, x_eval_0, num_steps=nfe)
  )
  lsd_samples[nfe] = np.asarray(
      flow_map.sample_forward_flow_map(lsd_fn, x_eval_0, num_steps=nfe)
  )
  metrics[f"distill_mse_vs_ode_{nfe}nfe"] = float(
      np.mean((distill_samples[nfe] - ode_ref) ** 2)
  )
  metrics[f"lsd_mse_vs_ode_{nfe}nfe"] = float(
      np.mean((lsd_samples[nfe] - ode_ref) ** 2)
  )

for k, v in metrics.items():
  print(f"{k:28s}: {v:.4f}")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(
    ode_ref[0, :, 0],
    "k-",
    lw=2,
    label=f"Teacher ODE ({config.teacher_ode_steps} NFE)",
)
for nfe in config.eval_num_steps:
  ax.plot(distill_samples[nfe][0, :, 0], "--", label=f"Distilled ({nfe} NFE)")
  ax.plot(lsd_samples[nfe][0, :, 0], ":", label=f"LSD ({nfe} NFE)")
ax.set_title("Forward Flow Map Generation vs. Teacher ODE")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(workdir, "flow_map_comparison.png"), dpi=100)
plt.show()